In [ ]:
import altair as alt
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Loading the combined csv
power_cut_df = pd.read_csv('power_cut_data.csv')

In [ ]:
power_cut_df.head()

In [ ]:
power_cut_df.describe()

In [ ]:
power_cut_df.dtypes

In [ ]:
power_cut_df['timestamp'] = pd.to_datetime(power_cut_df['timestamp'])

In [ ]:
power_cut_df.head()

In [ ]:
power_cut_df['Month_Sort'] = power_cut_df['timestamp'].dt.to_period('M')
power_cut_df['Month_Display'] = power_cut_df['timestamp'].dt.strftime('%B %Y')

In [ ]:
power_cut_df.head()

In [ ]:
# Group by the sorting period, display name, and site, then sum the power cuts
# grouped_df = power_cut_df.groupby(['Month_Sort', 'Month_Display', 'site_ID'])['power_cut_flag'].sum().reset_index()

# Pivot the data: Months become rows, Sites become columns, values are the power cuts
pivot_df = power_cut_df.pivot(index=['Month_Sort', 'Month_Display'], columns='site_ID', values='power_cut_flag').fillna(0)

# Sort chronologically using 'Month_Sort', then drop it so only 'Month_Display' shows on the x-axis
pivot_df = pivot_df.sort_index(level='Month_Sort')
pivot_df.index = pivot_df.index.get_level_values('Month_Display')

"""
# --- 3. Plotting the Chart ---
# Set up the stacked bar chart
ax = pivot_df.plot(kind='bar', stacked=True, figsize=(12, 7), colormap='tab20')

# Formatting the visual elements
plt.title('Total Power Cuts by Site (Minutes) [July 2025 - February 2026]', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12, fontweight='bold')
plt.ylabel('Total Power Cuts (Minutes)', fontsize=12, fontweight='bold')

# Rotate the "July 2025" labels so they don't overlap
# plt.xticks(rotation=45, ha='right')

# Move the legend outside the chart to prevent it from covering the bars
plt.legend(title='Sites', bbox_to_anchor=(1.02, 1), loc='upper left')

# Adjust layout to fit everything and render
plt.tight_layout()
plt.show()
"""
# --- 3. Plotting the Chart (WITH TOTALS & GRIDLINES) ---
ax = pivot_df.plot(kind='bar', stacked=True, figsize=(12, 7), colormap='tab20')

# Add horizontal gridlines behind the bars
# ax.grid(axis='y', linestyle='--', alpha=0.7, zorder=0)

# Calculate the total for each month (summing across the rows)
monthly_totals = pivot_df.sum(axis=1)

# Loop through the totals and place them on top of the bars
for i, total in enumerate(monthly_totals):
    y_offset = total + (max(monthly_totals) * 0.01)
    ax.text(i, y_offset, str(int(total)), ha='center', va='bottom', fontweight='bold', fontsize=11)

# Formatting the visual elements
plt.title('Total Power Cuts by Site (Minutes) [July 2025 - February 2026]', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12, fontweight='bold')
plt.ylabel('Total Power Cuts (Minutes)', fontsize=12, fontweight='bold')

# plt.xticks(rotation=45, ha='right')
plt.legend(title='Sites', bbox_to_anchor=(1.02, 1), loc='upper left')

# Extend the y-axis slightly to ensure the top numbers aren't cut off
plt.ylim(0, max(monthly_totals) * 1.1)

plt.tight_layout()
plt.show()

In [ ]:
# If your 'month' column is not already sorted or in a date format, you might want to define a custom sort order
# Example for string months:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

if 'power_cut_df' in locals() or 'power_cut_df' in globals():
    chart = alt.Chart(power_cut_df).mark_bar().encode(
        x=alt.X('month:N', sort=month_order if 'month_order' in locals() else None, title='Month'),
        y=alt.Y('count():Q', title='Count of Power Cut Flags'),
        color=alt.Color('power_cut_flag:N', title='Power Cut Flag (0=No, 1=Yes)', scale=alt.Scale(range=['green', 'red']))
    ).properties(
        title='Power Cut Flag by Site ID and Month'
    )

    # For a stacked bar chart by site_ID, if you want to stack power_cut_flag within each site and month:
    chart_stacked = alt.Chart(power_cut_df).mark_bar().encode(
        x=alt.X('month:N', sort=month_order if 'month_order' in locals() else None, title='Month'),
        y=alt.Y('count():Q', stack='normalize', title='Proportion of Power Cut Flags'),
        color=alt.Color('power_cut_flag:N', title='Power Cut Flag (0=No, 1=Yes)', scale=alt.Scale(range=['green', 'red'])),
        column=alt.Column('site_ID:N', header=alt.Header(titleOrient="bottom", labelOrient="bottom"), title='Site ID')
    ).properties(
        title='Stacked Power Cut Flag by Month per Site ID'
    )

    # To display the chart (uncomment the one you want to see):
    # chart.display()
    # chart_stacked.display()

    # If you want to see the individual bars stacked for each site_ID and month on one chart (less 'vertical' per se, but stacked):
    chart_vertical_stacked_single = alt.Chart(power_cut_df).mark_bar().encode(
        x=alt.X('month:N', title='Month', axis=alt.Axis(labels=True)),
        y=alt.Y('sum(power_cut_flag):Q', stack='zero', title='Total Power Cut Flags'),
        color=alt.Color('site_ID:N', title='Site ID'),
        order=alt.Order('site_ID:N'), # Order for stacking
        tooltip=['month', 'site_ID', 'sum(power_cut_flag)']
    ).properties(
        title='Vertical Stacked Bar Chart of Power Cut Flags per Month (Stacked by Site ID)'
    )
    chart_vertical_stacked_single.display()

else:
    print("Please provide your DataFrame name (e.g., 'df') and ensure it contains 'site_ID', 'month', and 'power_cut_flag' columns.")
    print("Example: your_dataframe = pd.DataFrame({'site_ID': ['A', 'B'], 'month': ['Jan', 'Feb'], 'power_cut_flag': [0, 1]})")